# Experiment: Stage1 Latent Clustering

Objective:
- Train only TriShift `stage1` and extract `z_mu` latent space.
- Use the stage1 latent space for clustering, UMAP, and label alignment analysis.
- Support two modes: `pbmc_celltype` and `trishift_condition`.

Notes:
- This notebook does not run `stage23`.
- Stage1 can optionally run a second scGPT-style ECS finetune phase after phase1 early stopping.
- PBMC data source is fixed to `scvi.data.pbmc_dataset()`.
- PBMC preprocessing is aligned to the scGPT integration example, but notebook-exposed options let you control gene-count filtering, total-count normalization, `log1p`, and HVG selection.
- Stage1 training uses the preprocessing output selected by the notebook: `X_log1p` when `log1p=True`, `X_normed` when normalization is enabled but `log1p=False`, otherwise raw `X`. No binning is used in the notebook PBMC path.
- PBMC `label_condition` is batch-like metadata from scvi PBMC, not perturbation condition.
- No cross-celltype holdout is done here; this notebook only handles clustering.
- Clustering prefers Leiden. If `python-igraph` is missing in the notebook environment, the helper falls back to `KMeans` automatically and still writes UMAP + metrics.
- UMAP plotting defaults are aligned to scGPT integration examples: `neighbors(use_rep=...)`, `min_dist=0.3`, compact `(4, 4)` figures, and `dpi=300` export.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
from IPython.display import Image, display

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import scripts.trishift.analysis.stage1_latent_clustering as stage1_latent_clustering
importlib.reload(stage1_latent_clustering)
run_stage1_latent_clustering = stage1_latent_clustering.run_stage1_latent_clustering

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Plan

- `pbmc_celltype`: load `scvi.data.pbmc_dataset()`, preprocess it with a local scGPT-style preprocessor, then cluster stage1 latent by PBMC `cell_type` and batch-like `condition`.
- `trishift_condition`: cluster stage1 latent by perturbation `condition` on a TriShift dataset.
- Default stage1 pool is `train_all_cells`, closer to scGPT-style cell embedding analysis.
- Outputs are written to `artifacts/stage1_latent_clustering/...`.


## Metrics guide

This notebook now reports both the original local metrics and scGPT/scIB-style metrics. For PBMC `label_cell_type`, if `scib` is installed, `ARI_cluster/label`, `NMI_cluster/label`, `ASW_label`, and `avg_bio` are taken from scIB; otherwise they fall back to the local implementations.

- `ari_leiden_vs_label` and `ARI_cluster/label`: the same Adjusted Rand Index between notebook-generated clusters and the target label. Higher is better. `1.0` means cluster assignments align almost perfectly; near `0` means close to random agreement.
- `nmi_leiden_vs_label` and `NMI_cluster/label`: the same Normalized Mutual Information between clusters and the target label. Higher is better. Compared with ARI, it is less sensitive to label permutation and easier to read when cluster counts differ.
- `silhouette_label` and `ASW_label`: the same silhouette score computed on the stage1 latent vectors using the target label itself, not the predicted cluster. Higher is better. Positive values mean same-label cells stay closer together than different-label cells; around `0` means weak separation; negative means labels are mixed.
- `avg_bio`: scGPT-style summary metric, defined as `mean(NMI_cluster/label, ARI_cluster/label, ASW_label)`. Higher is better. On PBMC `label_cell_type`, this now uses scIB when available, matching scGPT more closely.
- `n_unique_clusters` vs `n_unique_labels`: quick sanity check. If clusters are far more than labels, the partition may be over-fragmented; if far fewer, the latent may be under-separating biological groups.
- `cluster_method`: `leiden` if the environment has `igraph`; otherwise `kmeans_fallback`. Compare runs only when the clustering method is the same.

How to read the results:
- `pbmc_celltype`: prioritize `label_cell_type`, especially `avg_bio`, `ARI_cluster/label`, `NMI_cluster/label`, and `ASW_label`. `label_condition` is batch-like and secondary.
- On scvi PBMC, `stage1_deg_weight` is currently inactive because the dataset has no true `control/stimulated` perturbation condition; the helper records this in `run_meta.json`.
- If ECS finetune is enabled, stage1 first early-stops on the base VAE objective, then runs a second finetune phase with `base_loss + ecs_weight * ecs_loss`.
- `trishift_condition`: prioritize `label_condition`. `label_ctrl_pert` only tells you whether stage1 separates control from perturbed cells at a coarse level.
- UMAP is for qualitative structure inspection. Trust the numeric metrics first, then use UMAP to understand failure modes such as label overlap, fragmentation, or batch-like stripes.


In [ ]:
# Core config
mode = "pbmc_celltype"              # "pbmc_celltype" or "trishift_condition"
dataset_name = "pbmc"               # pbmc / adamson / dixit / norman / replogle_k562_essential / replogle_rpe1_essential
split_id = 1
stage1_pool_mode = "train_all_cells"  # "train_all_cells" or "ctrl_only"
random_seed = 24

# Stage1 training
stage1_epochs = 40        # None -> helper default
stage1_z_dim = 128         # None -> defaults.yaml model.stage1.z_dim
stage1_batch_size = 64           # None -> defaults.yaml
stage1_lr = 0.001         # None -> defaults.yaml
stage1_beta = None   # None -> defaults.yaml
stage1_sched_gamma = 0.9  # None -> defaults.yaml train.stage1.sched_gamma
stage1_deg_weight = 1         # None -> defaults.yaml; inactive on scvi PBMC because there is no perturbation condition
stage1_ecs_enable = True    # None -> defaults.yaml train.stage1.ecs.enable
stage1_ecs_epochs = 30        # None -> defaults.yaml train.stage1.ecs.epochs
stage1_ecs_lr = 0.001        # None -> defaults.yaml train.stage1.ecs.lr
stage1_ecs_sched_gamma = 0.9  # None -> defaults.yaml train.stage1.ecs.sched_gamma
stage1_ecs_weight = 10  # None -> defaults.yaml train.stage1.ecs.weight
stage1_ecs_threshold = 0.8    # None -> defaults.yaml train.stage1.ecs.threshold
stage1_ecs_patience = 10      # None -> defaults.yaml train.stage1.ecs.patience
stage1_ecs_min_delta = 0.0001  # None -> defaults.yaml train.stage1.ecs.min_delta
pbmc_source = "scvi"              # PBMC-only; fixed to scvi.data.pbmc_dataset()
pbmc_deg_mode = "by_cell_type"    # PBMC-only; kept for compatibility, currently inactive on scvi PBMC
pbmc_filter_gene_by_counts = 3  # 0/False -> disable gene-count filtering
pbmc_normalize_total = 1e4      # 0/False -> disable total-count normalization
pbmc_log1p = True             # False -> skip log1p and feed X_normed/raw X into stage1
pbmc_n_hvg = 1200            # 0/False -> disable HVG subsetting

# Latent + clustering
l2_normalize_latent = True
neighbors_k = None                # None -> scGPT-style scanpy default
leiden_resolution = 1.0
umap_min_dist = 0.3
reuse_z_mu_cache = False
save_dpi = 300
cluster_figsize = (4, 4)
label_figsize = (4, 4)
point_size_cluster = None        # None -> use scanpy/scGPT default
point_size_label = None          # None -> use scanpy/scGPT default
preview_width = 720

# PBMC-only config
pbmc_train_frac = 0.95



In [ ]:
run_kwargs = dict(
    mode=mode,
    dataset_name=dataset_name,
    split_id=split_id,
    stage1_pool_mode=stage1_pool_mode,
    random_seed=random_seed,
    stage1_epochs=stage1_epochs,
    stage1_z_dim=stage1_z_dim,
    stage1_batch_size=stage1_batch_size,
    stage1_lr=stage1_lr,
    stage1_beta=stage1_beta,
    stage1_sched_gamma=stage1_sched_gamma,
    stage1_deg_weight=stage1_deg_weight,
    stage1_ecs_enable=stage1_ecs_enable,
    stage1_ecs_epochs=stage1_ecs_epochs,
    stage1_ecs_lr=stage1_ecs_lr,
    stage1_ecs_sched_gamma=stage1_ecs_sched_gamma,
    stage1_ecs_weight=stage1_ecs_weight,
    stage1_ecs_threshold=stage1_ecs_threshold,
    stage1_ecs_patience=stage1_ecs_patience,
    stage1_ecs_min_delta=stage1_ecs_min_delta,
    pbmc_source=pbmc_source,
    pbmc_train_frac=pbmc_train_frac,
    pbmc_deg_mode=pbmc_deg_mode,
    pbmc_filter_gene_by_counts=pbmc_filter_gene_by_counts,
    pbmc_normalize_total=pbmc_normalize_total,
    pbmc_log1p=pbmc_log1p,
    pbmc_n_hvg=pbmc_n_hvg,
    l2_normalize_latent=l2_normalize_latent,
    neighbors_k=neighbors_k,
    leiden_resolution=leiden_resolution,
    umap_min_dist=umap_min_dist,
    reuse_z_mu_cache=reuse_z_mu_cache,
    save_dpi=save_dpi,
    cluster_figsize=cluster_figsize,
    label_figsize=label_figsize,
    point_size_cluster=point_size_cluster,
    point_size_label=point_size_label,
)

result = run_stage1_latent_clustering(**run_kwargs)
result.metrics_df


In [ ]:
print(f"Output dir: {result.out_dir}")
print(f"Latent h5ad: {result.latent_h5ad_path}")
print(f"z_mu cache: {result.z_mu_path}")
print(f"Metrics csv: {result.metrics_csv_path}")
print(f"Train logs: {result.train_logs_path}")
print(f"Cluster method: {result.metadata['cluster_method']}")
print()
print("Figure files:")
for key, fig_path in sorted(result.figure_paths.items()):
    print(f"- {key}: {fig_path}")


In [ ]:
result.metrics_df


In [ ]:
figure_map = result.figure_paths

for key in ["umap_by_cluster", *[k for k in figure_map if k.startswith("umap_by_") and k != "umap_by_cluster"]]:
    fig_path = figure_map.get(key)
    if fig_path:
        print(key)
        display(Image(filename=fig_path, width=preview_width))
